# Bronze Ingestion - DONKI Events

Ingests raw NASA DONKI space-weather API responses for the configured Artemis II mission window into the Bronze Delta table.

Table creation is handled by `notebooks/00_setup/01_create_bronze_tables.py.ipynb`; this notebook only appends new raw responses.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "orion" / "config.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break


In [ ]:
from src.orion.config import DONKI_BRONZE_COLUMNS
from src.orion.ingestion.donki_events import DEFAULT_EVENT_TYPES, ingest_donki_events, raw_donki_events_table_name

target_table = raw_donki_events_table_name()
print(f"Target table: {target_table}")
print(f"Event types: {', '.join(DEFAULT_EVENT_TYPES)}")
print(f"Expected columns: {', '.join(DONKI_BRONZE_COLUMNS)}")


In [ ]:
result = ingest_donki_events(spark, target_table=target_table)

print(f"Run ID: {result['ingestion_run_id']}")
print(f"Records requested: {result['records_requested']}")
print(f"Records written: {result['records_written']}")
print(f"Status codes: {result['response_status_codes']}")


In [ ]:
display(
    spark.sql(f"""
        SELECT
            ingestion_run_id,
            source_system,
            source_endpoint,
            event_type,
            response_status_code,
            length(response_body) AS response_body_length,
            response_hash,
            ingested_at,
            ingested_date,
            mission_name
        FROM {target_table}
        ORDER BY ingested_at DESC, event_type
        LIMIT 20
    """)
)
